# HCAD Building Other Features, 2021–2025

This notebook extends the 2021 processing of HCAD building_other.txt through 2025. The file contains building-level information for commercial and other income-producing properties, including building type, year built, building area, depreciation, quality, PGI classification, occupancy, and income-related variables.
Because one HCAD account may contain multiple buildings, the raw table cannot be merged directly into the property-level dataset. Numeric building characteristics are summarized across buildings within each account, while categorical variables used for rental-property identification, such as building type and PGI category, are retained separately as account-level indicators.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

In [2]:
project = Path(r"C:\Users\alys_\Documents\Research\Evictions")
hcad_root = project / "01_DATA_RAW" / "HCAD"
output_root = project / "02_DATA_PROCESSED" / "building_other"

output_root.mkdir(parents=True, exist_ok=True)

In [12]:
years = [2021, 2022, 2023, 2024, 2025]

num_cols = [
    "Depr_Val",
    "cama_replacement_cost",
    "accrued_depr_pct",
    "date_erected",
    "eff",
    "yr_remodel",
    "im_sq_ft",
    "act_ar",
    "heat_ar",
    "gross_ar",
    "eff_ar",
    "base_ar",
    "perimeter",
    "pct",
    "nra",
    "lease_rt",
    "occ_rt",
    "tot_inc"
]

apt_types = ["4209", "4211", "4212", "4214", "4221", "4222", "4299"]
apt_pgi = ["16", "17", "18", "19"]


In [13]:
def load_other(year):
    path = hcad_root / f"pdata{year}" / "building_other.txt"

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        encoding="latin1",
        low_memory=False
    )

    df.columns = df.columns.str.strip()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
                .replace("", pd.NA)
            )

    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    return df

In [14]:
def make_other(year):
    df = load_other(year)

    raw_rows = len(df)
    raw_accounts = df["acct"].nunique()

    df = df[
        df["acct"].notna() &
        df["bld_num"].notna()
    ].copy()

    for col in ["date_erected", "eff", "yr_remodel"]:
        df.loc[
            ~df[col].between(1800, year),
            col
        ] = np.nan

    df["apartment_type"] = df["impr_tp"].isin(apt_types)
    df["apartment_pgi"] = df["category"].isin(apt_pgi)
    df["manufactured_type"] = df["impr_tp"].eq("4213")
    df["manufactured_pgi"] = df["category"].eq("13")
    df["has_remodel"] = df["yr_remodel"].notna()

    base = (
        df.groupby("acct")
        .agg(
            n_buildings_other=("bld_num", "nunique"),
            apartment_type=("apartment_type", "max"),
            apartment_pgi=("apartment_pgi", "max"),
            manufactured_type=("manufactured_type", "max"),
            manufactured_pgi=("manufactured_pgi", "max"),
            has_remodel=("has_remodel", "max"),
            year_built=("date_erected", "mean"),
            effective_year=("eff", "mean"),
            remodel_year=("yr_remodel", "mean"),
            depr_pct=("accrued_depr_pct", "mean")
        )
        .reset_index()
    )

    sums = (
        df.groupby("acct")[
            [
                "im_sq_ft",
                "act_ar",
                "nra",
                "cama_replacement_cost",
                "tot_inc"
            ]
        ]
        .sum(min_count=1)
        .reset_index()
        .rename(
            columns={
                "im_sq_ft": "improvement_area",
                "act_ar": "actual_area",
                "nra": "net_rentable_area",
                "cama_replacement_cost": "replacement_cost",
                "tot_inc": "total_income"
            }
        )
    )

    out = base.merge(
        sums,
        on="acct",
        how="left",
        validate="one_to_one"
    )

    missing = (
        out.drop(columns="acct")
        .isna()
        .mean()
        .mul(100)
        .round(2)
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "feature"})
    )

    missing["year"] = year

    summary = {
        "year": year,
        "raw_rows": raw_rows,
        "raw_accounts": raw_accounts,
        "final_accounts": len(out),
        "lost_accounts": raw_accounts - len(out),
        "apartment_type": out["apartment_type"].sum(),
        "apartment_pgi": out["apartment_pgi"].sum(),
        "manufactured_type": out["manufactured_type"].sum(),
        "manufactured_pgi": out["manufactured_pgi"].sum()
    }

    out.to_parquet(
        output_root / f"building_other_{year}.parquet",
        index=False
    )

    return summary, missing

Numeric variables were aggregated according to their meaning rather than with one universal rule. Building years and depreciation percentages were averaged across valid buildings, while physical area, replacement cost, net rentable area, and income were summed because they represent quantities that accumulate across buildings. Zero or temporally impossible year values were treated as missing before aggregation. Apartment and manufactured-housing classifications were retained as indicators if any building within the account met the corresponding HCAD classification.

In [15]:
summary = []
missing = []

for year in years:
    print("processing", year)

    s, m = make_other(year)

    summary.append(s)
    missing.append(m)

    gc.collect()

processing 2021
processing 2022
processing 2023
processing 2024
processing 2025


In [16]:
df_summary_other = pd.DataFrame(summary)

In [17]:
df_missing_other = pd.concat(
    missing,
    ignore_index=True
)
display(df_summary_other)

,year,raw_rows,raw_accounts,final_accounts,lost_accounts,apartment_type,apartment_pgi,manufactured_type,manufactured_pgi
0,2021,153908,66731,66731,0,5997,2731,295,0
1,2022,155141,67337,67337,0,6038,2855,298,0
2,2023,156761,68084,68084,0,6101,2931,299,0
3,2024,158144,68319,68319,0,6163,3008,299,0
4,2025,159110,68850,68850,0,6189,3029,310,110


In [18]:
display(
    df_missing_other.pivot(
        index="feature",
        columns="year",
        values="missing_pct"
    )
)

year,2021,2022,2023,2024,2025
feature,,,,,
actual_area,0.00,0.00,0.00,0.00,0.00
apartment_pgi,0.00,0.00,0.00,0.00,0.00
apartment_type,0.00,0.00,0.00,0.00,0.00
depr_pct,0.00,0.00,0.00,0.00,0.00
effective_year,0.01,0.00,0.00,0.00,0.00
has_remodel,0.00,0.00,0.00,0.00,0.00
improvement_area,0.00,0.00,0.00,0.00,0.00
manufactured_pgi,0.00,0.00,0.00,0.00,0.00
manufactured_type,0.00,0.00,0.00,0.00,0.00


In [19]:
df_other_2022 = pd.read_parquet(
    output_root / "building_other_2022.parquet"
)

df_other_2023 = pd.read_parquet(
    output_root / "building_other_2023.parquet"
)

df_other_2024 = pd.read_parquet(
    output_root / "building_other_2024.parquet"
)

df_other_2025 = pd.read_parquet(
    output_root / "building_other_2025.parquet"
)

In [20]:
display(df_other_2022.head())

display(
    df_other_2022.describe().T
)

,acct,n_buildings_other,apartment_type,apartment_pgi,manufactured_type,manufactured_pgi,has_remodel,year_built,effective_year,remodel_year,depr_pct,improvement_area,actual_area,net_rentable_area,replacement_cost,total_income
0,0010020000016,1,False,False,False,False,True,1900.0,2013.0,2021.0,0.150,22608,24112,0,0.0,NaN
1,0010040000001,1,False,False,False,False,False,1923.0,1966.0,NaN,0.500,22236,48370,0,0.0,NaN
2,0010100000001,1,False,False,False,False,False,1985.0,1985.0,NaN,0.488,4898,4898,0,0.0,NaN
3,0010120000010,1,False,False,False,False,False,2000.0,2000.0,NaN,0.380,744404,838480,0,0.0,NaN
4,0010150000001,1,False,False,False,False,True,1870.0,1965.0,2015.0,0.500,54818,54818,38356,0.0,901366.0


,count,mean,std,min,25%,50%,75%,max
n_buildings_other,67337.0,2.108633e+00,5.003495e+00,1.0,1.00,1.000000e+00,2.000000e+00,3.130000e+02
year_built,67337.0,1.985285e+03,2.282478e+01,1845.0,1970.00,1.985000e+03,2.004333e+03,2.022000e+03
effective_year,67334.0,1.989669e+03,2.067607e+01,1845.0,1977.50,1.992800e+03,2.006000e+03,2.022000e+03
remodel_year,12788.0,2.013132e+03,6.222086e+00,1950.0,2010.00,2.014000e+03,2.018000e+03,2.022000e+03
depr_pct,67337.0,4.041061e-01,2.184204e-01,0.0,0.24,4.102000e-01,5.500000e-01,9.997000e-01
improvement_area,67337.0,3.625973e+04,1.881188e+05,90.0,3000.00,6.428000e+03,1.965600e+04,1.786540e+07
actual_area,67337.0,3.985877e+04,1.929853e+05,90.0,3440.00,7.450000e+03,2.214200e+04,1.786739e+07
net_rentable_area,67337.0,1.744815e+05,2.008857e+06,0.0,0.00,0.000000e+00,0.000000e+00,2.774451e+08
replacement_cost,67322.0,1.624696e+03,4.654668e+04,0.0,0.00,0.000000e+00,0.000000e+00,8.148408e+06
total_income,11630.0,1.583452e+07,7.394344e+07,0.0,279580.25,1.030052e+06,6.438035e+06,3.132355e+09


## Results and decisions
The building_other methodology was applied consistently to HCAD data from 2021 through 2025. Raw observations increased from 153,908 in 2021 to 159,110 in 2025, representing 66,731 and 68,850 unique HCAD accounts respectively. No accounts were lost during processing.
Because an account may contain multiple buildings, numeric characteristics were aggregated according to their meaning. Valid building years and depreciation percentages were averaged across buildings, while physical area, replacement-cost, and income quantities were aggregated across the buildings belonging to an account. Year values below 1800 or later than the corresponding tax year were treated as invalid and converted to missing before aggregation.
Apartment building-type counts increased gradually from 5,997 accounts in 2021 to 6,189 in 2025, while apartment PGI classifications increased from 2,731 to 3,029. Manufactured-housing building classifications were stable across years, but PGI category 13 was absent from 2021–2024 and appeared for 110 accounts in 2025. This is treated as a possible change in HCAD coding rather than evidence of a sudden change in the housing stock.
Remodel year and total income remain substantially incomplete. Zero-valued fields are retained as provided by HCAD, so a low formal missingness percentage does not necessarily imply that a field contains substantive information for every property. Feature usefulness will therefore be evaluated again after the complete property dataset and rental cohort are constructed.